# League of Legends Esports Win Probability — Elo Modelling

**LCK and LEC, 2024–2026** | Oracle’s Elixir match data

This notebook reconstructs three chronological Elo models:
- **V1**: separate historical team names; K=24; no seasonal regression.
- **V2**: canonical team identity mapping; K=24; no seasonal regression.
- **V3**: V2 plus 75% carryover of prior-year Elo deviation from 1500.

**Target:** probability the Blue-side team wins each individual game. Blue side itself receives **no bonus**.

**Evaluation caution:** 2025 was repeatedly used in model development, and 2026 was inspected during development. Report the results as **exploratory**, not a blind holdout. Training data for any given Elo prediction is strictly earlier match results, in timestamp order. All games in 2024 were included for rating initialization.

**Inputs:** download original yearly CSVs (2024, 2025, 2026) from [Oracle’s Elixir Downloads](https://master.d36liwrx5rvjnc.amplifyapp.com/tools/downloads) and place in the same directory as this notebook. Original large CSVs are not embedded here.


In [1]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

YEARS = [2024, 2025, 2026]
INPUTS = {year: Path(f"{year}_LoL_esports_match_data_from_OraclesElixir.csv") for year in YEARS}
for year, path in INPUTS.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Download the CSV from Oracle's Elixir first.")
print('All three yearly input CSVs found.')

All three yearly input CSVs found.


## 1. Load and validate source data

Each game in the source is represented by 10 player rows and 2 team rows. Only team rows are required. A valid completed match has two team rows, one Blue and one Red, and exactly one winning team.

In [2]:
cols = ['gameid','date','league','teamname','side','position','result']
df = pd.concat([pd.read_csv(INPUTS[y], usecols=cols, low_memory=False) for y in YEARS], ignore_index=True)
team_df = df.loc[df['position'].eq('team')].copy()
summary = team_df.groupby('gameid').agg(team_rows=('result','size'), winner_count=('result','sum'), blue_count=('side',lambda s:(s=='Blue').sum()), red_count=('side',lambda s:(s=='Red').sum()))
valid_ids = summary.index[(summary.team_rows==2)&(summary.winner_count==1)&(summary.blue_count==1)&(summary.red_count==1)]
clean = team_df.loc[team_df.gameid.isin(valid_ids)].copy()
print('Raw rows:',len(df),'| Raw games:',df.gameid.nunique())
print('Invalid games:',len(summary)-len(valid_ids),'| Valid games:',len(valid_ids))

Raw rows: 361260 | Raw games: 30105
Invalid games: 1 | Valid games: 30104


## 2. One row per game; choose LCK and LEC

In [3]:
blue = clean.loc[clean.side=='Blue',['gameid','date','league','teamname','result']].rename(columns={'teamname':'blue_team_original','result':'blue_win'})
red = clean.loc[clean.side=='Red',['gameid','teamname']].rename(columns={'teamname':'red_team_original'})
matches = blue.merge(red,on='gameid',how='inner',validate='one_to_one')
matches['date'] = pd.to_datetime(matches['date'])
matches['year'] = matches.date.dt.year
matches = matches.loc[matches.league.isin(['LCK','LEC'])].sort_values(['date','gameid']).reset_index(drop=True)
assert len(matches)==2524
assert not matches.gameid.duplicated().any()
print(matches.groupby(['year','league']).size())

year  league
2024  LCK       482
      LEC       294
2025  LCK       555
      LEC       308
2026  LCK       502
      LEC       383
dtype: int64


## 3. Map renamed team identities

Two continuity mappings are used for Elo identity: MAD Lions KOI → Movistar KOI; Team BDS → Shifters. Historical names are preserved in their own columns. Rogue and Natus Vincere are **not** merged; the latter represents an ownership/slot acquisition rather than a simple rebrand. This is an analytical assumption about organizational continuity, **not** an assertion of unchanged rosters. The canonical name may refer to a later brand for historical matches.

In [4]:
TEAM_MAP = {'MAD Lions KOI':'Movistar KOI','Team BDS':'Shifters'}
for side in ['blue','red']:
    matches[f'{side}_team_canonical'] = matches[f'{side}_team_original'].replace(TEAM_MAP)
print('Games:',len(matches),'| Historical team labels:',len(set(matches.blue_team_original)|set(matches.red_team_original)))

Games: 2524 | Historical team labels: 25


## 4. Chronological Elo predictions

Elo starts at 1500. Each league has an independent rating system. Pre-match probability is recorded **before** the observed result updates either team. For V3, a team's rating is regressed to 1500 once at its first observed match of each new calendar year: `1500 + 0.75 * (old_rating - 1500)`. Team names in the data are assumed correct and modelled as given.

K=24 and α=0.75 were chosen by exploratory 2025 comparisons; they are not objectively optimal or independently confirmed.

In [5]:
def predict_elo(data, *, canonical=False, k=24, alpha=1.0):
    ratings = {}
    last_year = {}
    records = []
    side_cols = ('blue_team_canonical','red_team_canonical') if canonical else ('blue_team_original','red_team_original')
    for row in data.sort_values(['date','gameid']).itertuples(index=False):
        bkey = (row.league, getattr(row,side_cols[0]))
        rkey = (row.league, getattr(row,side_cols[1]))
        for key in (bkey,rkey):
            if key not in ratings:
                ratings[key] = 1500.0
                last_year[key] = row.year
            elif last_year[key] != row.year:
                ratings[key] = 1500.0 + alpha*(ratings[key]-1500.0)
                last_year[key] = row.year
        b, r = ratings[bkey], ratings[rkey]
        p = 1/(1+10**((r-b)/400))
        records.append((row.gameid,p,b,r))
        ratings[bkey] = b + k*(row.blue_win-p)
        ratings[rkey] = r + k*(p-row.blue_win)
    return pd.DataFrame(records,columns=['gameid','prob','blue_elo_before','red_elo_before'])

for name,canonical,alpha in [('v1',False,1.0),('v2',True,1.0),('v3',True,.75)]:
    forecast = predict_elo(matches,canonical=canonical,k=24,alpha=alpha)
    matches = matches.merge(forecast[['gameid','prob']].rename(columns={'prob':f'{name}_blue_prob'}),on='gameid',validate='one_to_one')
assert matches[['v1_blue_prob','v2_blue_prob','v3_blue_prob']].notna().all().all()
print('V1, V2 and V3 predictions:',len(matches))

V1, V2 and V3 predictions: 2524


## 5. Compare model performance (exploratory evaluation)

Accuracy thresholds probability at 50%. Lower log loss and Brier score indicate better probabilistic predictions. These 2025 and 2026 measurements do **not** represent a fully untouched holdout, because results informed the development process.

In [6]:
rows=[]
for year in [2025,2026]:
    subset=matches[matches.year==year]
    for name in ['v1','v2','v3']:
        p=subset[f'{name}_blue_prob']
        y=subset.blue_win
        rows.append({'Year':year,'Model':name.upper(),'Matches':len(subset),'Accuracy':accuracy_score(y,p>=0.5),'Log Loss':log_loss(y,p),'Brier Score':brier_score_loss(y,p)})
model_summary=pd.DataFrame(rows)
display(model_summary.style.format({'Accuracy':'{:.2%}','Log Loss':'{:.4f}','Brier Score':'{:.4f}'}))

,Year,Model,Matches,Accuracy,Log Loss,Brier Score
0,2025,V1,863,67.21%,0.6139,0.2121
1,2025,V2,863,67.21%,0.6140,0.2122
2,2025,V3,863,66.98%,0.6132,0.2122
3,2026,V1,885,63.05%,0.6395,0.2242
4,2026,V2,885,63.39%,0.6388,0.2237
5,2026,V3,885,63.50%,0.6355,0.2227


## 6. Diagnostic breakdown: V3 by league

In [7]:
rows=[]
for league,subset in matches[matches.year==2026].groupby('league'):
    p=subset.v3_blue_prob
    y=subset.blue_win
    rows.append({'League':league,'Matches':len(subset),'Accuracy':accuracy_score(y,p>=.5),'Log Loss':log_loss(y,p),'Brier Score':brier_score_loss(y,p)})
league_summary=pd.DataFrame(rows)
display(league_summary.style.format({'Accuracy':'{:.2%}','Log Loss':'{:.4f}','Brier Score':'{:.4f}'}))

,League,Matches,Accuracy,Log Loss,Brier Score
0,LCK,502,63.15%,0.6374,0.2238
1,LEC,383,63.97%,0.6330,0.2213


## 7. Export results for the app / documentation

`latest_team_ratings.csv` stores **post-match** Elo ratings through the last available game of each team. For prospective games, this is only a snapshot as of the data end date; predictions will become stale when new matches occur. Cross-league probabilities are not supported because league Elo scales are independently estimated.

In [8]:
model_summary.to_csv('lol_elo_model_comparison.csv',index=False)
export_cols=['gameid','date','year','league','blue_team_original','red_team_original','blue_team_canonical','red_team_canonical','blue_win','v1_blue_prob','v2_blue_prob','v3_blue_prob']
matches[export_cols].to_csv('lol_match_predictions_final.csv',index=False)
# Obtain end-of-sample V3 ratings by rerunning the same chronological procedure.
def final_ratings(data,k=24,alpha=.75):
    ratings,last_year={},{}
    for row in data.sort_values(['date','gameid']).itertuples(index=False):
        bk=(row.league,row.blue_team_canonical)
        rk=(row.league,row.red_team_canonical)
        for key in (bk,rk):
            if key not in ratings:
                ratings[key]=1500.0;last_year[key]=row.year
            elif last_year[key]!=row.year:
                ratings[key]=1500+alpha*(ratings[key]-1500);last_year[key]=row.year
        b,r=ratings[bk],ratings[rk]
        p=1/(1+10**((r-b)/400))
        ratings[bk]=b+k*(row.blue_win-p)
        ratings[rk]=r+k*(p-row.blue_win)
    return pd.DataFrame([{'league':league,'team':team,'elo_rating':rating} for (league,team),rating in ratings.items()]).sort_values(['league','team'])
end_ratings=final_ratings(matches)
end_ratings.to_csv('latest_team_ratings.csv',index=False)
print('Saved 3 CSV files. Elo ratings for',len(end_ratings),'league/team identities.')

Saved 3 CSV files. Elo ratings for 23 league/team identities.


## Interpretation and limitations

- 2026 V3: approximately **63.50% accuracy**, **0.6355 log loss**, and **0.2227 Brier score** across 885 games (verify from the executed table).
- Improvements against V1 are small and are **exploratory**, not evidence of statistically reliable generalisation.
- The calendar-year Elo carryover is a simplified proxy for offseason roster changes; it does not model individual player transfers or patches.
- The Oracle's Elixir dataset may use revised/harmonised team labels; no independent historical timestamp audit of source corrections has been conducted.
- Predictions use the known result only after the match when updating subsequent ratings. No bookmaker odds or profitability/backtesting analysis has been performed.
- In an app, only match two teams from the **same league**, and clearly label ratings as **through September 2026** (per the available LCK/LEC records).
